# Lấy nội dung trang chưa crawl qua web.archive.org (bản lưu Internet Archive)

**Chuẩn bị:** Settings → Internet **On**, chỉ cần CPU (không cần GPU). Add Input: dataset chứa `targets.parquet`
(tạo bằng `slug_targets.py`). Chạy tiếp lần trước: Add Input thêm Output của lần đó (có `records.jsonl`).

Mỗi trang tải đúng 1 lần qua `https://web.archive.org/web/<YEAR_HINT>id_/<URL gốc>` (bản lưu gần nhất, HTML gốc
không có thanh công cụ Wayback), tải chậm và tự nghỉ khi bị báo quá tải (429/503).

**Kết quả trong Output:** `ids_<đầu>_<cuối>_synced_wayback.parquet` (đúng định dạng crawl, dùng với `merge_ids.py`
và pipeline) và `provenance.csv` (id, URL gốc, URL bản lưu, thời điểm lưu) để trích dẫn nguồn với BTC.
**Chạy bằng Save Version** để tắt tab được.

In [ ]:
# ===== CẤU HÌNH =====
WORKERS = 4              # số luồng tải song song
DELAY = 1.0              # giây nghỉ giữa 2 lần tải của MỖI luồng (tổng khoảng WORKERS/DELAY trang/giây)
YEAR_HINT = '2025'       # web.archive.org trả bản lưu gần mốc này nhất
MAX_PAGES = None         # None = tất cả; 50 = chạy thử
TIME_LIMIT_HOURS = 11.0  # dừng an toàn trước giới hạn 12 giờ của Kaggle
MIN_CHARS = 200          # bài trích ra ngắn hơn thì coi như rỗng
USER_AGENT = 'Mozilla/5.0 (compatible; AncientWings-R2AI2026-research; +https://web.archive.org)'


In [ ]:
import glob, json, os, re, subprocess, sys, threading, time
START = time.time()
WORK = '/kaggle/working'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'trafilatura'], check=True)
import pyarrow as pa, pyarrow.parquet as pq, requests, trafilatura

found = sorted(glob.glob('/kaggle/input/**/targets.parquet', recursive=True))
if not found:
    raise FileNotFoundError('Chưa gắn dataset chứa targets.parquet (Add Input).')
targets = pq.read_table(found[0]).to_pylist()
print('Trang cần tải:', len(targets), '| từ', found[0])

RECORDS = f'{WORK}/records.jsonl'
done = {}
for path in sorted(glob.glob('/kaggle/input/**/records.jsonl', recursive=True)) + [RECORDS]:
    if os.path.exists(path):
        with open(path, encoding='utf-8') as f:
            for line in f:
                try:
                    r = json.loads(line)
                except ValueError:
                    continue
                if r.get('status') in ('ok', 'not_archived', 'empty', 'homepage_redirect'):
                    done[r['id']] = r
if done and not os.path.exists(RECORDS):
    with open(RECORDS, 'w', encoding='utf-8') as f:  # giữ kết quả lần trước trong output lần này
        for r in done.values():
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
todo = [t for t in targets if t['id'] not in done]
if MAX_PAGES:
    todo = todo[:MAX_PAGES]
print('Đã có từ lần trước:', len(done), '| còn phải tải:', len(todo))


In [ ]:
STAMP = re.compile(r'/web/(\d{14})id_/(.*)$')
pause_until = [0.0]
lock = threading.Lock()


def original_path(url):
    return re.sub(r'^https?://[^/]+', '', url).split('?')[0] or '/'


def fetch(session, target):
    url = target['url']
    record = {'id': target['id'], 'url': url}
    for attempt in range(5):
        wait = pause_until[0] - time.time()
        if wait > 0:
            time.sleep(wait)
        try:
            resp = session.get(f'https://web.archive.org/web/{YEAR_HINT}id_/{url}', timeout=(20, 60),
                               allow_redirects=True)
        except requests.RequestException as error:
            record['status'] = f'error:{type(error).__name__}'
            time.sleep(5 * (attempt + 1))
            continue
        if resp.status_code in (429, 503):
            with lock:  # cả nhóm luồng nghỉ, để không bị chặn
                pause_until[0] = max(pause_until[0], time.time() + 60 * (attempt + 1))
            record['status'] = f'error:http{resp.status_code}'
            continue
        if resp.status_code == 404:
            record['status'] = 'not_archived'
            return record
        if resp.status_code != 200:
            record['status'] = f'error:http{resp.status_code}'
            time.sleep(5)
            continue
        match = STAMP.search(resp.url)
        record['archive_url'] = resp.url
        record['archive_timestamp'] = match.group(1) if match else None
        archived_target = match.group(2) if match else url
        if original_path(archived_target) == '/' and original_path(url) != '/':
            record['status'] = 'homepage_redirect'
            return record
        resp.encoding = resp.apparent_encoding if not resp.encoding or resp.encoding.lower() == 'iso-8859-1' else resp.encoding
        html = resp.text
        doc = trafilatura.bare_extraction(html, url=url, include_comments=False, include_tables=True,
                                          favor_recall=True, with_metadata=True)
        if doc is None:
            text, title = '', ''
        elif isinstance(doc, dict):
            text, title = doc.get('text') or '', doc.get('title') or ''
        else:
            text, title = getattr(doc, 'text', '') or '', getattr(doc, 'title', '') or ''
        record['title'] = title
        record['text'] = text
        record['status'] = 'ok' if len(text) >= MIN_CHARS else 'empty'
        return record
    return record


def worker(items, out, counter):
    session = requests.Session()
    session.headers['User-Agent'] = USER_AGENT
    for target in items:
        if time.time() - START > TIME_LIMIT_HOURS * 3600:
            return
        record = fetch(session, target)
        with lock:
            out.write(json.dumps(record, ensure_ascii=False) + '\n')
            out.flush()
            counter[record.get('status', 'error')] = counter.get(record.get('status', 'error'), 0) + 1
        time.sleep(DELAY)


counter = {}
with open(RECORDS, 'a', encoding='utf-8') as out:
    threads = [threading.Thread(target=worker, args=(todo[i::WORKERS], out, counter), daemon=True)
               for i in range(WORKERS)]
    for t in threads:
        t.start()
    while any(t.is_alive() for t in threads):
        time.sleep(60)
        total = sum(counter.values())
        rate = total / max(1e-9, time.time() - START)
        print(f'{total}/{len(todo)} | {counter} | {rate:.2f} trang/giây | còn ~{(len(todo) - total) / max(rate, 1e-9) / 60:.0f} phút', flush=True)
print('Xong lượt này:', counter)


In [ ]:
import hashlib
latest = {}
with open(RECORDS, encoding='utf-8') as f:
    for line in f:
        try:
            r = json.loads(line)
        except ValueError:
            continue
        if r['id'] not in latest or latest[r['id']].get('status') != 'ok':
            latest[r['id']] = r
ok = sorted((r for r in latest.values() if r.get('status') == 'ok'), key=lambda r: r['id'])
statuses = {}
for r in latest.values():
    statuses[r.get('status')] = statuses.get(r.get('status'), 0) + 1
print('Trạng thái:', statuses, '| chưa thử:', len(targets) - len(latest))
if ok:
    rows = [{'id': r['id'], 'url': r['url'], 'final_url': r['archive_url'], 'title': r.get('title') or '',
             'text': r['text'], 'domain': r['url'].split('/')[2], 'extractor': 'trafilatura_wayback',
             'lang': 'vi', 'status': 'ok', 'quality_flags': ['wayback'], 'published_at': None,
             'text_sha256': hashlib.sha256(r['text'].encode('utf-8')).hexdigest(),
             'archive_timestamp': r.get('archive_timestamp')} for r in ok]
    name = f"{WORK}/ids_{ok[0]['id']}_{ok[-1]['id']}_synced_wayback.parquet"
    pq.write_table(pa.Table.from_pylist(rows), name, compression='zstd')
    with open(f'{WORK}/provenance.csv', 'w', encoding='utf-8') as f:
        f.write('id,original_url,archive_url,archive_timestamp\n')
        for r in ok:
            f.write(f"{r['id']},{r['url']},{r['archive_url']},{r.get('archive_timestamp') or ''}\n")
    print('Ghi', name, '| trang có nội dung:', len(ok))
    print('provenance.csv: nguồn gốc từng trang (dùng để trích dẫn với BTC)')
if len(latest) < len(targets):
    print('CHƯA XONG: Save Version lại, gắn Output lần này làm Input để tải tiếp.')
